# Experiment 11: the neural blend gate, run locally

**Only needed if the gate did not run inside `10_neural_kaggle.ipynb`.** That
notebook runs the same comparison on Kaggle when
`lgbm_bag08_seedblend5_oof.npy` is attached as a dataset. If it was attached, the
answer is already in the Kaggle output and this notebook is redundant.

The fallback path is: download `neural_oof.npy` and `neural_test.npy` from the
Kaggle notebook's output, drop them in `artifacts/oof/`, and Run All here.

## What is being decided

Whether a neural model earns a place in the final blend. Not whether its CV is
good, which it almost certainly is not. `06` and `07` established the rule the
expensive way: gate on the measured rank-blend AUC against the best single model,
never on rank correlation, which was anti-predictive in both directions here.

The bar is the **within-family blending floor of +0.000072** from `07`, cleared on
**every fold**. The 5-seed seed blend cleared its own comparison at +0.000546 on
5/5 folds with a paired sd of 0.000094, which is what a real blend gain looks like
on this data.

Both a 50/50 blend and a weight curve are reported. 50/50 is the primary gate
because it is the one weight not chosen after seeing the answer. The curve exists
because a partner that is weaker overall can still help at a small weight, and a
50/50-only test would call that a failure.

In [1]:
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

SEED, N_SPLITS = 42, 5
TARGET, ID = "addicted_label", "id"

# The floor from 07: the best within-family LightGBM pair was worth +0.000072.
# A different model family has to beat that to have earned its complexity.
FLOOR = 0.000072
LGB_BEST_CV = 0.963880      # exp 15, the 5-seed bagged blend

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "data" / "raw" / "train.csv").exists())
RAW = ROOT / "data" / "raw"
OOF = ROOT / "artifacts" / "oof"

LGB_OOF_PATH = ROOT / "artifacts" / "kaggle_upload" / "lgbm_bag08_seedblend5_oof.npy"
NN_OOF_PATH = OOF / "neural_oof.npy"
NN_TEST_PATH = OOF / "neural_test.npy"
LGB_SUB_PATH = ROOT / "submissions" / "lgbm_bag08_seedblend5.csv"

have_nn = NN_OOF_PATH.exists()
print(f"root {ROOT}")
for p in (LGB_OOF_PATH, NN_OOF_PATH, NN_TEST_PATH, LGB_SUB_PATH):
    print(f"  [{'found' if p.exists() else 'MISSING'}] {p.name}")
if not have_nn:
    print()
    print("neural_oof.npy is not here. Download it from the Kaggle notebook output")
    print(f"into {OOF} and re-run. Nothing below will produce a number without it.")

root smartphone-addiction
  [found] lgbm_bag08_seedblend5_oof.npy
  [MISSING] neural_oof.npy
  [MISSING] neural_test.npy
  [found] lgbm_bag08_seedblend5.csv

neural_oof.npy is not here. Download it from the Kaggle notebook output
into smartphone-addiction/artifacts/oof and re-run. Nothing below will produce a number without it.


In [2]:
train = pd.read_csv(RAW / "train.csv")
y = train[TARGET].to_numpy().astype(np.float32)

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
# int64 explicitly: the sha below is taken over the raw bytes, so a platform where
# `int` means int32 would fail the check for no real reason.
folds = np.full(len(train), -1, dtype=np.int64)
for i, (_, va) in enumerate(skf.split(train, y)):
    folds[va] = i

# Same gate as 10_neural_kaggle.ipynb cell 2. If the Kaggle run produced its OOF
# under a different split, the vector cannot be blended with anything on disk here
# and every number below would be meaningless rather than merely wrong.
EXPECT = {"rows": 691369, "rate": 0.709424, "sha": "ec282b0968059676",
          "sizes": [138274, 138274, 138274, 138274, 138273],
          "first20": [3, 3, 3, 4, 2, 3, 4, 0, 3, 4, 1, 1, 2, 1, 3, 1, 1, 4, 0, 3]}
got = {"rows": len(train), "rate": round(float(y.mean()), 6),
       "sha": hashlib.sha256(folds.tobytes()).hexdigest()[:16],
       "sizes": np.bincount(folds).tolist(), "first20": folds[:20].tolist()}

ALIGNED = True
for k in EXPECT:
    ok = got[k] == EXPECT[k]
    ALIGNED &= ok
    print(f"  [{'ok' if ok else 'MISMATCH'}] {k}")
    if not ok:
        print(f"        expected {EXPECT[k]}")
        print(f"        got      {got[k]}")

# The neural OOF has to be the right length and cover every row. A vector of zeros
# in some fold is what a partially failed Kaggle run leaves behind, and it would
# blend cleanly and silently.
if have_nn:
    nn_oof = np.load(NN_OOF_PATH)
    len_ok = len(nn_oof) == len(train)
    ALIGNED &= len_ok
    print(f"  [{'ok' if len_ok else 'MISMATCH'}] neural oof length {len(nn_oof):,}")
    if len_ok:
        dead = [f for f in range(N_SPLITS)
                if np.ptp(nn_oof[folds == f]) < 1e-12]
        ALIGNED &= not dead
        print(f"  [{'ok' if not dead else 'MISMATCH'}] every fold has a real "
              f"prediction{'' if not dead else f', constant in folds {dead}'}")

print(f"\nalignment: {'verified' if ALIGNED else 'FAILED, stop and reconcile'}")

  [ok] rows
  [ok] rate
  [ok] sha
  [ok] sizes
  [ok] first20

alignment: verified


## The gate

Everything below is paired: each weight is scored on the identical rows, fold by
fold, against the LightGBM blend alone. Paired differences are far more precise
than the absolute AUCs, which is the whole reason fold spread is the wrong
yardstick for this comparison. That amendment is written up.

In [3]:
if not (have_nn and ALIGNED):
    print("gate not run: neural OOF missing or alignment failed")
else:
    lgb_oof = np.load(LGB_OOF_PATH)

    rank = lambda v: pd.Series(v).rank(pct=True).to_numpy()
    per_fold = lambda v: np.array([roc_auc_score(y[folds == f], v[folds == f])
                                   for f in range(N_SPLITS)])
    r_lgb, r_nn = rank(lgb_oof), rank(nn_oof)
    l = per_fold(lgb_oof)
    nn_fold = per_fold(nn_oof)

    def score(w):
        """w is the neural share. Returns blend mean, gain, paired sd, folds won."""
        b = per_fold((1 - w) * r_lgb + w * r_nn)
        d = b - l
        return b.mean(), d.mean(), d.std(ddof=1), int((d > 0).sum())

    print(f"LightGBM blend alone : {l.mean():.6f}  (ledger exp 15: {LGB_BEST_CV:.6f})")
    print(f"neural alone         : {nn_fold.mean():.6f} +/- {nn_fold.std():.6f}")
    print(f"Spearman             : {float(np.corrcoef(r_lgb, r_nn)[0, 1]):.4f} "
          f"(descriptive only, anti-predictive on this data)")

    b50, g50, s50, w50 = score(0.5)
    print(f"\n50/50 blend          : {b50:.6f}   gain {g50:+.6f}, "
          f"paired sd {s50:.6f}, wins {w50}/{N_SPLITS} folds")

    # Weights are scored on the same out-of-fold vector they are chosen from, so
    # the best one is an upper bound rather than an estimate of leaderboard value.
    print("\nweight curve, neural share (chosen on OOF, so optimistic):")
    grid = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50]
    rows = [(w, *score(w)) for w in grid]
    best_gain = max(r[2] for r in rows)
    for w, bm, g, sd, wins in rows:
        flag = "  <-- best" if g == best_gain else ""
        print(f"  w={w:.2f}  blend {bm:.6f}  gain {g:+.6f}  "
              f"paired sd {sd:.6f}  wins {wins}/{N_SPLITS}{flag}")

    w_best, b_best, g_best, sd_best, wins_best = max(rows, key=lambda r: r[2])

    print()
    if g50 > FLOOR and w50 == N_SPLITS:
        VERDICT, W_SUBMIT = "proceed", 0.5
        print("VERDICT: proceed. The blend clears the floor on every fold at the one")
        print("weight that was not chosen after seeing the answer. Submit it.")
    elif g_best > FLOOR and wins_best == N_SPLITS:
        VERDICT, W_SUBMIT = "marginal", w_best
        print(f"VERDICT: marginal, and weight-sensitive. 50/50 does not clear the")
        print(f"floor but w={w_best:.2f} gains {g_best:+.6f} on {wins_best}/{N_SPLITS} "
              f"folds, paired sd {sd_best:.6f}.")
        print("That weight was picked on this same OOF, so treat it as an upper")
        print("bound. Worth one submission to test against the LB, not worth a week.")
    else:
        VERDICT, W_SUBMIT = "stop", None
        print(f"VERDICT: stop. No weight clears the floor of {FLOOR:+.6f} on every")
        print("fold. The neural family does not help either, which closes the last")
        print("diversity hypothesis on the board. Write it into this repo and spend")
        print("the remaining time on the flywheel artifacts.")

gate not run: neural OOF missing or alignment failed


## Build the submission, only if the gate said so

A stop verdict still gets an `experiments.csv` row. A rejected idea that leaves no
trace is the one failure mode the ledger exists to prevent, and this one cost a GPU
session to establish.

In [4]:
if not (have_nn and ALIGNED) or W_SUBMIT is None:
    print("no submission built, which is the correct outcome for a stop verdict")
else:
    lgb_sub = pd.read_csv(LGB_SUB_PATH)
    nn_test = np.load(NN_TEST_PATH)
    assert len(lgb_sub) == len(nn_test), "test length mismatch, do not submit this"

    rank = lambda v: pd.Series(v).rank(pct=True).to_numpy()
    out = lgb_sub.copy()
    out[TARGET] = ((1 - W_SUBMIT) * rank(lgb_sub[TARGET].to_numpy())
                   + W_SUBMIT * rank(nn_test))
    tag = f"{int(round(W_SUBMIT * 100)):02d}"
    path = ROOT / "submissions" / f"lgbm_seedblend5_neural_w{tag}.csv"
    out.to_csv(path, index=False)
    print(f"wrote {path.name}, {len(out):,} rows, w_neural={W_SUBMIT:.2f}")
    print(out.head())

print()
print("experiments.csv row to append once the LB score comes back:")
if have_nn and ALIGNED:
    print(f"  cv_mean for the neural model alone : {nn_fold.mean():.6f}")
    print(f"  cv_std                             : {nn_fold.std():.6f}")
    print(f"  verdict                            : {VERDICT}")

no submission built, which is the correct outcome for a stop verdict

experiments.csv row to append once the LB score comes back:


## What this changed

Filled in after the run and promoted the working notes either way. A neural model
that fails to blend is a real result: it closes the last hypothesis on the board
and redirects the remaining time to the four flywheel artifacts, which are worth
more than another 0.001 in either direction.